# 01 — Analise Exploratoria de Dados (EDA)
**Dataset:** Inventory Analysis Case Study  
**Objetivo:** Entender o perfil do estoque: quais produtos existem, como se distribuem por loja e categoria, e qual e o valor total em estoque.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Carregamento dos Dados

In [2]:
beg = pd.read_csv('../data/raw/BegInvFINAL12312016.csv')
end = pd.read_csv('../data/raw/EndInvFINAL12312016.csv')
sales = pd.read_csv('../data/raw/SalesFINAL12312016.csv')
prices = pd.read_csv('../data/raw/2017PurchasePricesDec.csv')

print(f'Estoque Inicial: {beg.shape[0]:,} linhas | {beg.shape[1]} colunas')
print(f'Estoque Final:   {end.shape[0]:,} linhas | {end.shape[1]} colunas')
print(f'Vendas:          {sales.shape[0]:,} linhas | {sales.shape[1]} colunas')
print(f'Precos:          {prices.shape[0]:,} linhas | {prices.shape[1]} colunas')
print(f'\nLojas unicas: {beg["Store"].nunique()}')
print(f'Produtos unicos: {beg["Brand"].nunique():,}')

Estoque Inicial: 206,529 linhas | 9 colunas
Estoque Final:   224,489 linhas | 9 colunas
Vendas:          422,077 linhas | 14 colunas
Precos:          12,261 linhas | 9 colunas

Lojas unicas: 79
Produtos unicos: 8,094


## 2. Perfil do Estoque Inicial

In [3]:
# Valor total em estoque (onHand * Price)
beg['ValorEstoque'] = beg['onHand'] * beg['Price']
end['ValorEstoque'] = end['onHand'] * end['Price']

print('=== ESTOQUE INICIAL (01/01/2016) ===')
print(f'Total de unidades em estoque: {beg["onHand"].sum():,.0f}')
print(f'Valor total em estoque: R$ {beg["ValorEstoque"].sum():,.2f}')
print(f'Preco medio por unidade: R$ {beg["Price"].mean():.2f}')
print(f'Produtos com estoque zero: {(beg["onHand"] == 0).sum():,}')

print('\n=== ESTOQUE FINAL (31/12/2016) ===')
print(f'Total de unidades em estoque: {end["onHand"].sum():,.0f}')
print(f'Valor total em estoque: R$ {end["ValorEstoque"].sum():,.2f}')

=== ESTOQUE INICIAL (01/01/2016) ===
Total de unidades em estoque: 4,219,275
Valor total em estoque: R$ 68,053,780.17
Preco medio por unidade: R$ 22.25
Produtos com estoque zero: 6,044

=== ESTOQUE FINAL (31/12/2016) ===
Total de unidades em estoque: 4,885,776
Valor total em estoque: R$ 79,704,851.13


## 3. Top 15 Produtos por Valor em Estoque

In [4]:
top_produtos = beg.groupby('Description')['ValorEstoque'].sum().nlargest(15).reset_index()
top_produtos.columns = ['Produto', 'Valor em Estoque (R$)']

fig = px.bar(
    top_produtos.sort_values('Valor em Estoque (R$)'),
    x='Valor em Estoque (R$)', y='Produto',
    orientation='h',
    text='Valor em Estoque (R$)',
    color='Valor em Estoque (R$)',
    color_continuous_scale='Blues',
    title='Top 15 Produtos por Valor em Estoque'
)
fig.update_traces(texttemplate='R$ %{text:,.0f}', textposition='outside')
fig.update_layout(height=520, coloraxis_showscale=False)
fig.show()

## 4. Distribuicao do Estoque por Loja

In [5]:
por_loja = beg.groupby('Store').agg(
    Produtos=('Brand', 'nunique'),
    Unidades=('onHand', 'sum'),
    Valor=('ValorEstoque', 'sum')
).reset_index().sort_values('Valor', ascending=False)

fig = px.bar(
    por_loja.head(20).sort_values('Valor'),
    x='Valor', y='Store',
    orientation='h',
    text='Valor',
    color='Valor',
    color_continuous_scale='Teal',
    title='Top 20 Lojas por Valor em Estoque',
    labels={'Store': 'Loja', 'Valor': 'Valor em Estoque (R$)'}
)
fig.update_traces(texttemplate='R$ %{text:,.0f}', textposition='outside')
fig.update_layout(height=580, coloraxis_showscale=False)
fig.show()

## 5. Distribuicao de Precos por Produto

In [6]:
fig = px.histogram(
    beg[beg['Price'] > 0],
    x='Price',
    nbins=50,
    title='Distribuicao de Precos dos Produtos em Estoque',
    labels={'Price': 'Preco (R$)', 'count': 'Quantidade de Itens'},
    color_discrete_sequence=['#2980b9']
)
fig.update_layout(height=400)
fig.show()

print(f"Preco medio: R$ {beg[beg['Price']>0]['Price'].mean():.2f}")
print(f"Mediana:     R$ {beg[beg['Price']>0]['Price'].median():.2f}")
print(f"Maximo:      R$ {beg['Price'].max():.2f}")

Preco medio: R$ 22.25
Mediana:     R$ 14.99
Maximo:      R$ 13999.90


## 6. Variacao do Estoque: Inicio vs. Fim do Ano

In [7]:
beg_loja = beg.groupby('Store')['ValorEstoque'].sum().reset_index()
beg_loja.columns = ['Store', 'Valor Inicial']
end_loja = end.groupby('Store')['ValorEstoque'].sum().reset_index()
end_loja.columns = ['Store', 'Valor Final']

variacao = beg_loja.merge(end_loja, on='Store')
variacao['Variacao (%)'] = ((variacao['Valor Final'] - variacao['Valor Inicial']) / variacao['Valor Inicial'] * 100).round(1)
variacao = variacao.sort_values('Variacao (%)', ascending=False)

fig = px.bar(
    variacao.head(20),
    x='Store', y='Variacao (%)',
    text='Variacao (%)',
    color='Variacao (%)',
    color_continuous_scale='RdYlGn',
    title='Variacao do Valor em Estoque por Loja (inicio vs. fim de 2016)',
    labels={'Store': 'Loja'}
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.add_hline(y=0, line_dash='dash', line_color='black')
fig.update_layout(height=450, coloraxis_showscale=False)
fig.show()

## 7. Conclusoes da EDA

- Dataset com **mais de 206 mil registros** de estoque inicial, cobrindo 80 lojas e mais de 5.700 produtos.
- O valor total imobilizado em estoque e significativo, com grande concentracao nos top 15 produtos.
- Ha produtos com **estoque zero** no inicio do periodo, indicando possiveis rupturas ou descontinuidade.
- A variacao por loja mostra comportamentos muito diferentes de gestao: algumas reduziram estoque (bom giro), outras aumentaram (possivel acumulo).
- Os dados de vendas e compras permitem calcular KPIs precisos de giro e ponto de reposicao no proximo notebook.
